In [ ]:
# 系统提示词
# 系统提示词就是省去每次的SystemMessage,直接在定义阶段就用prompt定义
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 1.加载环境变量
load_dotenv()

# 2.初始化模型
model = init_chat_model(
        "deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}}
    )

# 3.定义tool（可以没有，不过这个时候只能对话，不算完整的Agent）
from langchain.tools import tool

# 定义工具默认需要三个信息
# - 工具名称：函数名
# - 工具输入：函数入参
# - 工具作用：函数的注释

# 三种定义方式
# - 基于@tool装饰器属性
# - 基于函数的文档注释
# - 基于BaseModel作为schema

# =================方式一：基于装饰器描述工具==================
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

tools = [tool1]

# =================方式二：基于文档注释描述工具==================
# 推荐
# 相较于方式一,更方便描述多参数
from langchain.tools import tool

# 定义一个查询天气的tool
@tool
def get_weather2(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    Get current weather and optional forecast.
    Args:
        location: city name or coordinates
        units: unit of degrees
        include_forecast: does it include the weather forecast
    """
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

tools = [tool1,get_weather2]

# ==================方式三：基于BaseModel描述工具参数=================
# 相较于方式二,更方便描述多参数且有约束的情况
from pydantic import BaseModel, Field
from typing import Literal
from langchain.tools import tool
# 例如一个查询天气的tool
class WeatherInput(BaseModel):
    """查询天气的输入参数."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(# Literal字面量 units只能是摄氏度和华氏度二选一
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )
    
# 定义一个查询天气的tool
@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result


tools = [tool1,get_weather]

# 4.创建智能体
agent = create_agent(
    model = model,
    tools=tools,
)

# 5.流式调用
stream = agent.stream_events(
    {"messages": [HumanMessage(content="杭州接下来几天的天气如何")]},
    version="v3"
)

for message in stream.messages:
    for text in message.text:
        print(text, end="", flush=True)

NameError: name '基于' is not defined